![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 29 -- Lab 1: Pretrained Encoders in torchvision and the Hub

A first look at pretrained models, the way a competition notebook starts: you load ResNet-18 with its published ImageNet weights from torchvision, read its two parts and its parameter count, look inside the weights file, run its preprocessing contract on Kaggle flower photos, see what its 1,000-class head says about flowers it has no class for, cut the head off to get the encoder's 512-number feature vector, and then do the same look-up on the Hugging Face Hub with `transformers` and `timm`.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `alxmamaev/flowers-recognition` (about 230 MB), downloaded by `kagglehub` in the setup cell: `flowers/<class>/*.jpg`, 4,317 photos in five folders (daisy, dandelion, rose, sunflower, tulip). Nothing is trained in this lab, so a CPU runtime is enough; the models download about 150 MB of weights on first use.

---
# Setup

In [ ]:
# kagglehub fetches the Kaggle dataset; timm is the third model library of the deck. Colab has torch, torchvision and transformers.
!pip install -q kagglehub timm

In [ ]:
import os, random
import kagglehub
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, " torch:", torch.__version__)

In [ ]:
path = kagglehub.dataset_download("alxmamaev/flowers-recognition")
ROOT = os.path.join(path, "flowers")
CLASSES = sorted(os.listdir(ROOT))                                   # folder names, alphabetical
counts = {c: len(os.listdir(os.path.join(ROOT, c))) for c in CLASSES}
photos = {c: os.path.join(ROOT, c, sorted(os.listdir(os.path.join(ROOT, c)))[0]) for c in CLASSES}   # one photo per class
print(counts)
print(photos["daisy"])

---
# Part 1 -- The model and its two parts

The *Every Classifier Has Two Parts* and *Pretrained on ImageNet-1k* slides. A pretrained model is an architecture plus a weights file; the encoder holds almost all the weights, the head is one `Linear`.

## Task 1: Load the pretrained ResNet-18

Create `weights = ResNet18_Weights.DEFAULT` and `model = resnet18(weights=weights)`, then put the model in inference mode with `model.eval()`. Print `weights.meta["_metrics"]["ImageNet-1K"]` (the published accuracies), `len(weights.meta["categories"])` (how many classes it knows) and `n_params = sum(p.numel() for p in model.parameters())`. Compare the three numbers with the *torchvision.models* table on the slides.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert n_params == 11_689_512 and len(weights.meta['categories']) == 1000, 'resnet18(weights=ResNet18_Weights.DEFAULT): 11,689,512 parameters and 1,000 ImageNet classes'
print('Task 1 passed')

## Task 2: The two parts

Loop over `model.named_children()` and print each child's name with its parameter count, `sum(p.numel() for p in child.parameters())`. Then compute `n_head`, the parameters of `model.fc`, and `n_encoder = n_params - n_head`, and print the share of the parameters that sits in the encoder. Which child holds most of the weights?

Syntax hint (the shape of the call, not the answer):

```python
for name, child in model.named_children():
    print(name, sum(p.numel() for p in child.parameters()))
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert n_head == 513_000 and n_encoder == 11_176_512, 'the head fc = Linear(512, 1000) has 513,000 parameters; everything else is the encoder'
print('Task 2 passed')

## Task 3: Inside the weights file

Take `sd = model.state_dict()` and print `len(sd)` (how many tensors the file holds) and the shapes of `sd["conv1.weight"]`, `sd["layer4.1.conv2.weight"]` and `sd["fc.weight"]`. Then draw the 64 first-layer kernels: `k = sd["conv1.weight"]` has shape (64, 3, 7, 7); scale each kernel to 0..1 with `(k - k.min()) / (k.max() - k.min())`, and show them in an 8 x 8 grid of `imshow` panels with `k[i].permute(1, 2, 0)` (channels last) and the axes switched off. They are Day 22's kernels, learned instead of written by hand.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(sd) == 122 and tuple(sd['conv1.weight'].shape) == (64, 3, 7, 7) and tuple(sd['fc.weight'].shape) == (1000, 512), 'state_dict: 122 tensors; conv1.weight (64, 3, 7, 7); fc.weight (1000, 512)'
print('Task 3 passed')

---
# Part 2 -- The preprocessing contract and the prediction

The *In Code: Load, Preprocess, Predict*, *The Preprocessing Contract* and *What the Pretrained Classifier Says About Your Photos* slides. The weights expect photos resized to 256, cropped to 224 and normalised with ImageNet's mean and std; `weights.transforms()` is that contract written down.

## Task 4: Run the contract on one photo

Create `preprocess = weights.transforms()` and print it. Open the sunflower photo, `img = Image.open(photos["sunflower"]).convert("RGB")`, print `img.size`, then compute `x = preprocess(img).unsqueeze(0)` and print `x.shape`, `x.min()`, `x.max()` and `x.mean()`. The slide's numbers: a shape of (1, 3, 224, 224) and values from about -2.1 to 2.6.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(x.shape) == (1, 3, 224, 224) and x.min() < -1.5 and x.max() > 1.5, 'preprocess(img).unsqueeze(0): (1, 3, 224, 224), normalised values well below 0 and above 1'
print('Task 4 passed')

## Task 5: Top-3 predictions for one photo per class

Write `predict_top(model, img, k=3)` that preprocesses the photo, runs the model under `torch.no_grad()`, turns the scores into probabilities with `softmax(dim=1)` and returns a list of `(label, probability)` pairs for the `k` most likely classes, using `weights.meta["categories"]` for the names. Run it on the five photos in `photos` and show them side by side with the folder name and the top-3 list as the title (`fontsize=8`). Which folders does ResNet-18 have a class for, and what does it say about the others?

Syntax hint (the shape of the call, not the answer):

```python
with torch.no_grad():
    probs = model(x).softmax(dim=1)[0]
top = probs.topk(k)              # top.values and top.indices
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert top3['daisy'][0][0] == 'daisy' and top3['daisy'][0][1] > 0.9, 'predict_top on the daisy photo: daisy with a probability above 0.9 (daisy is an ImageNet class)'
print('Task 5 passed')

## Task 6: How often is the folder name the answer?

For every class, take `random.sample` of 20 file names from its folder and count how many times the top-1 label returned by `predict_top(..., k=1)` is exactly the folder name. Print one line per class with the count out of 20. Only `daisy` is an ImageNet class; the other four cannot be answered by this head however good the encoder is.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert hits['tulip'] == 0 and hits['rose'] == 0 and hits['daisy'] >= 5, 'the 1,000-class head has a daisy row and no tulip or rose row: it cannot answer those folders'
print('Task 6 passed')

---
# Part 3 -- The encoder's output

The *Encoder's Output: 512 Numbers per Photo* slide. Replace the head with `nn.Identity()` and the model returns the pooled feature vector instead of class scores.

## Task 7: Cut the head off

Build `encoder = resnet18(weights=weights)`, set `encoder.fc = nn.Identity()` and call `encoder.eval()`. Stack the preprocessed five photos into one batch `xb` of shape (5, 3, 224, 224) with `torch.stack`, run the encoder under `torch.no_grad()` and store the result as `feats`. Print `feats.shape`, the minimum and maximum of the daisy's vector `feats[0]`, and how many of its 512 numbers are exactly zero. Then draw `feats[0].view(16, 32)` with `imshow` as on the slide.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(feats.shape) == (5, 512) and feats.min() >= 0, 'encoder with fc = nn.Identity(): one 512-number vector per photo, none negative (they leave a ReLU)'
print('Task 7 passed')

## Task 8: Two daisies and a tulip

Take a second daisy photo, `sorted(os.listdir(os.path.join(ROOT, "daisy")))[1]`, compute its feature vector `v2` with the encoder, and print two distances with `torch.dist`: between the first daisy `feats[0]` and `v2`, and between the first daisy and the tulip `feats[4]`. The slide says a second daisy gives a similar pattern and a tulip a different one; is that what the distances show? (Day 31 turns this into image search.)

In [ ]:
# Your code here


---
# Part 4 -- The same weights on the Hugging Face Hub

The *What the Hub Is*, *In Code: The Hub in Three Lines* and *Three Doors to the Same Weights* slides. The Hub is a repository per model; `transformers` loads it by id, `timm` is the third door.

## Task 9: The pipeline, three lines

Import `pipeline` from `transformers`, create `clf = pipeline("image-classification", model="microsoft/resnet-50")` (about 100 MB of weights on first use) and run it on the five photos. For each class print the folder name and the two most likely labels with their scores, `clf(img)[:2]`. Compare with Task 5: same training data, a bigger network.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert hub_top['daisy'][0]['label'] == 'daisy', 'pipeline("image-classification", model="microsoft/resnet-50") on the daisy photo: daisy first'
print('Task 9 passed')

## Task 10: The explicit way

Import `AutoImageProcessor` and `AutoModelForImageClassification`, load both from `"microsoft/resnet-50"`, and print the processor (its size, crop fraction, mean and std are the contract, written in `preprocessor_config.json`). Run the daisy photo through `processor(images=img, return_tensors="pt")`, print the shape of `inputs["pixel_values"]`, compute `logits = hf_model(**inputs).logits` under `torch.no_grad()`, print its shape and the label `hf_model.config.id2label[logits.argmax(-1).item()]`. Match each step with its torchvision twin from Part 2.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(logits.shape) == (1, 1000) and hf_label == 'daisy', 'AutoModelForImageClassification: (1, 1000) logits; id2label of the argmax is daisy'
print('Task 10 passed')

## Task 11: Browse the Hub from code

Import `list_models` from `huggingface_hub` and fetch the eight most downloaded image-classification models: `list(list_models(pipeline_tag="image-classification", sort="downloads", limit=8))`. Print each model's `id` and `downloads`. Which families from the torchvision table do you recognise, and what does `in1k` in a `timm/` name mean?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(top_models) == 8 and all(m.downloads > 0 for m in top_models), 'list_models(pipeline_tag="image-classification", sort="downloads", limit=8): eight models with download counts'
print('Task 11 passed')

## Task 12: The third door: timm, with a new head in one argument

Import `timm` and create `tm = timm.create_model("resnet18", pretrained=True, num_classes=5)`: the same ResNet-18 weights from the Hub (`timm/resnet18.a1_in1k`), with the head already replaced for five classes. Print `tm.fc` and the total parameter count, and compare the count with Task 1: the encoder is the same, the head shrank from 513,000 to 2,565 parameters. This is exactly what Lab 2 builds by hand with torchvision.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert isinstance(tm.fc, nn.Linear) and tm.fc.out_features == 5 and n_timm == 11_179_077, 'timm.create_model("resnet18", pretrained=True, num_classes=5): fc = Linear(512, 5); 11,176,512 + 2,565 parameters'
print('Task 12 passed')

---
## Task 13: Reflect

Task 2 put 96% of ResNet-18's parameters in the encoder. Which part would you keep for the flowers and which would you replace, and why? In Task 6 the tulip folder scored 0 out of 20 although the encoder sees tulips well: where exactly is the information missing? Task 4 printed values from about -2 to 2.6: what would happen to the predictions if you fed the model the raw 0 to 255 pixels, and would Python warn you? Name two things a Hub model card tells you that the weights file alone does not. In an offline contest the models are handed over as folders: which of the three doors would you open, and with which call?

*Your answers here*

---
## Conclusion

- **Two parts**: ResNet-18 loaded with its published weights, 11,176,512 parameters in the encoder and 513,000 in the head; the weights file read as a `state_dict` and the learned first-layer kernels drawn.
- **The preprocessing contract**: `weights.transforms()` applied to a photo, the normalised tensor's shape and range checked against the slide.
- **A pretrained head answers with the classes it knows**: `daisy` recognised, tulips and roses not, because the 1,000-class head has no row for them.
- **The encoder's output**: the head replaced by `nn.Identity()`, one 512-number vector per photo.
- **The Hub**: the same kind of model loaded with `pipeline`, with `AutoImageProcessor` and `AutoModelForImageClassification`, browsed with `list_models`, and `timm` creating a ResNet-18 with a five-class head in one call.

---

Made By **Sattam Altwaim**